# Bayes Computational Project
## Group Members:
- **Benjamin Nelson**: Wrote a majority of the code relating to the SKLearn versions of each Bayes model
- **Harrison Yeboah**: Wrote the implementation for the Bernoulli Bayes model
- **Han Nguyen**: Wrote the implementation for the Gaussian Bayes model
- **Duc Le**: Worked on markdown explanations + analysis
- **Kaiden McCready**: Worked on markdown explanations + analysis 
## Project Explanation
## Data
Initially, we ran our model on the Iris dataset as a proof of concept. We now run our KNN algorithm on:
### Auto MPG Dataset
The Auto MPG Dataset from the UCI Machine Learning Repository contains data about different vehicles. The dataset includes information on displacement, HP, weight, and acceleration. With our KNN algorithm we use this data to predict the fuel efficiency of vehicles.

In [2]:
from pathlib import Path
from urllib.request import urlretrieve
import numpy as np
from tabulate import tabulate

path = Path("mnist.npz")
if not path.exists():
    urlretrieve(
        "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz",
        path
    )

with np.load(path) as data:
    x_train = data["x_train"]
    y_train = data["y_train"]
    x_test = data["x_test"]
    y_test = data["y_test"]

print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)

(60000, 28, 28) (60000,)
(10000, 28, 28) (10000,)


In [ ]:
digits = [3, 5]

train_mask = np.isin(y_train, digits)
test_mask = np.isin(y_test, digits)

X_train = x_train[train_mask].reshape(-1, 784).astype(float) # flatten dataset
Y_train = y_train[train_mask]

X_test = x_test[test_mask].reshape(-1, 784).astype(float)
Y_test = y_test[test_mask]

In [3]:
# within 3s, find mean and stdev
threes = X_train[Y_train == 3]

mean_3 = threes.mean(axis=0)
std_3 = threes.std(axis=0)
var_3 = threes.var(axis=0)


In [4]:
# within 5s, find mean and stdev
fives = X_train[Y_train == 5]

mean_5 = fives.mean(axis=0)
std_5 = fives.std(axis=0)
var_5 = fives.var(axis=0)

In [5]:
# smoothing variance

epsilon = 1e-9 * max(var_3.max(), var_5.max())
var_3_smooth = var_3 + epsilon
var_5_smooth = var_5 + epsilon

In [6]:
# prior 
prior_3 = np.mean(Y_train == 3)
prior_5 = np.mean(Y_train == 5)

print(prior_3, prior_5)
print(prior_3 + prior_5)  # should be 1.0 - yay it is!

0.5307306094182825 0.46926939058171746
1.0


gaussian_score finds the posterior based on the discriminate function likelihood (normal dist) * prior

In [7]:
def gaussian_score(x, mean, variance, prior):
    pixel_scores = -1/2 * (np.log(2 * np.pi * variance) + (x - mean) ** 2 / variance)
    return np.log(prior) + pixel_scores.sum()

In [8]:
# list comprehension, check all images and do predictions
predictions = np.array([
    3 if gaussian_score(x, mean_3, var_3_smooth, prior_3)
         > gaussian_score(x, mean_5, var_5_smooth, prior_5)
    else 5
    for x in X_test
])

In [9]:
labels = [3, 5]

confusion = np.array([
    [
        np.sum((Y_test == actual) & (predictions == predicted))
        for predicted in labels
    ]
    for actual in labels
])

headers = ["", "Predicted 3", "Predicted 5"]
data = [
    ["Actual 3", confusion[0, 0], f"{confusion[0, 1]}"],
    ["Actual 5", f"{confusion[1, 0]}", confusion[1, 1]]
]

print(tabulate(data, headers=headers, tablefmt="grid"))

+----------+---------------+---------------+
|          |   Predicted 3 |   Predicted 5 |
+==========+===============+===============+
| Actual 3 |           976 |            34 |
+----------+---------------+---------------+
| Actual 5 |           280 |           612 |
+----------+---------------+---------------+


In [7]:
TP = confusion[1, 1]  # actual 5, predicted 5
FP = confusion[0, 1]  # actual 3, predicted 5
FN = confusion[1, 0]  # actual 5, predicted 3

accuracy = np.trace(confusion) / confusion.sum()
recall = TP / (FN + TP)
precision = TP / (FP + TP)

f1 = 2 / (1/precision + 1/recall)

print(accuracy) #np.trace gives the sum of diagonal
print(f1)

0.8922187171398528
0.8786264061574895


In [13]:
#Self contained sklearn Normal(gaussian) and Bernoulli Niave Bayes
from pathlib import Path
from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import BernoulliNB
from urllib.request import urlretrieve
import numpy as np
from tabulate import tabulate


def NiaveBayesSKLearn(diget1, diget2, isBernoulli = False):
    #Loading data
    path = Path("mnist.npz")
    if not path.exists():
        urlretrieve(
            "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz",
            path
    )
    with np.load(path) as data:
        x_train = data["x_train"]
        y_train = data["y_train"]
        x_test = data["x_test"]
        y_test = data["y_test"]

    #Filtering out for our comparison
    digits = [diget1, diget2]

    train_mask = np.isin(y_train, digits)
    test_mask = np.isin(y_test, digits)

    X_train = x_train[train_mask].reshape(-1, 784).astype(float) # flatten dataset
    Y_train = y_train[train_mask]

    X_test = x_test[test_mask].reshape(-1, 784).astype(float)
    Y_test = y_test[test_mask]

    #Locality rule
    frame = 0
    #Assingment of either Gaussian or Bernoulli
    if not isBernoulli:
        frame = GaussianNB()
    else:
        frame = BernoulliNB()

    #fit predict
    frame = frame.fit(X_train,Y_train)
    frame_y_predicts = frame.predict(X_test)

    #Confusion Matrix
    labels = [diget1, diget2]
    confusion = np.array([
        [
            np.sum((Y_test == actual) & (frame_y_predicts == predicted))
            for predicted in labels
        ]
        for actual in labels
    ])

    #Making Table
    headers = ["", "Predicted " + str(diget1), "Predicted "  + str(diget2)]
    data = [
        ["Actual " + str(diget1), confusion[0, 0], f"{confusion[0, 1]}"],
        ["Actual " + str(diget2), f"{confusion[1, 0]}", confusion[1, 1]]
    ]

    #returning table
    return(tabulate(data, headers=headers, tablefmt="grid"))

In [12]:
#Test NiaveBayesSKLearn
print(NiaveBayesSKLearn(3,5,True))
print(NiaveBayesSKLearn(3,5,False))

+----------+---------------+---------------+
|          |   Predicted 3 |   Predicted 5 |
+==========+===============+===============+
| Actual 3 |           955 |            55 |
+----------+---------------+---------------+
| Actual 5 |           150 |           742 |
+----------+---------------+---------------+
+----------+---------------+---------------+
|          |   Predicted 3 |   Predicted 5 |
+==========+===============+===============+
| Actual 3 |           976 |            34 |
+----------+---------------+---------------+
| Actual 5 |           280 |           612 |
+----------+---------------+---------------+


In [ ]:
# Bernoulli needs to be binarized here
X_train_bin = (X_train > 127).astype(float)
X_test_bin = (X_test > 127).astype(float)

In [ ]:
# Smoothing done here
theta_3 = (X_train_bin[Y_train == 3].sum(axis=0) + 1) / (np.sum(Y_train == 3) + 2)

In [ ]:
# within 5s, per-pixel probability of being "on" here
theta_5 = (X_train_bin[Y_train == 5].sum(axis=0) + 1) / (np.sum(Y_train == 5) + 2)

In [ ]:
# bernoulli score function
def bernoulli_score(x, theta, prior):
    pixel_scores = x * np.log(theta) + (1 - x) * np.log(1 - theta)
    return np.log(prior) + pixel_scores.sum()

In [ ]:
# Doing predictions 
bern_predictions = np.array([
    3 if bernoulli_score(x, theta_3, prior_3)
         > bernoulli_score(x, theta_5, prior_5)
    else 5
    for x in X_test_bin
])

In [ ]:
# bernoulli confusion matrix
bern_confusion = np.array([
    [
        np.sum((Y_test == actual) & (bern_predictions == predicted))
        for predicted in labels
    ]
    for actual in labels
])

data = [
    ["Actual 3", bern_confusion[0, 0], bern_confusion[0, 1]],
    ["Actual 5", bern_confusion[1, 0], bern_confusion[1, 1]]
]

print(tabulate(data, headers=headers, tablefmt="grid"))

In [ ]:
# Confusion matrix
TP = bern_confusion[1, 1]  # actual 5, predicted 5
FP = bern_confusion[0, 1]  # actual 3, predicted 5
FN = bern_confusion[1, 0]  # actual 5, predicted 3

bern_accuracy = np.trace(bern_confusion) / bern_confusion.sum()
bern_recall = TP / (FN + TP)
bern_precision = TP / (FP + TP)

bern_f1 = 2 / (1/bern_precision + 1/bern_recall)

print(bern_accuracy)
print(bern_f1)